## 微调

In [ ]:
[{'instruction': '你是一个文本实体识别领域的专家，你需要从给定的句子中提取目的地;出发地;返程地;餐饮;住宿;产品;交通;预算;时长;时间;人群;强度;人气;人数;天气.以json格式输出,如{"entity_text":"北京","entity_label":"目的地"}注意:1.输出的每一行都必须是正确的json字符串.2.找不到任何实体时,输出"没有找到任何实体".',
  'input': '文本:五一，我要去泉州旅游，想要感受一下宋元时期中国的世界海洋商贸盛况，计划去3天2晚，住宿预算大概是300~400元每晚，希望住在市中心，能帮我规划一下路线吗?',
  'output': '{"entity_text":"泉州","entity_label":"目的地"}{"entity_text":"市中心","entity_label":"住宿"}{"entity_text":"300~400元","entity_label":"预算"}{"entity_text":"五一","entity_label":"时间"}{"entity_text":"3天2晚","entity_label":"时长"}{"entity_text":"宋元时期中国的世界海洋商贸盛况","entity_label":"产品"}'}]

In [ ]:
!swift sft \
--model models/Qwen2.5-7B-Instruct \
--model_type qwen2_5 \
--dataset train_e.json \
--val_dataset valid_e.json \
--logging_steps 5 \
--save-steps 10 \
--learning_rate 3e-5 \
--output_dir output \
--lora_rank 8 \
--lora_alpha 16 \
--lora_dropout 0. \
--weight_decay 0.01 \
--num_train_epochs 5 \
--per_device_train_batch_size 2 \
--gradient_accumulation_steps 2 \
--warmup-ratio 0.01 \
--eval_steps 20 \
--evaluation-strategy steps \
--max-grad-norm 1.0 \
--lr-scheduler-type cosine \
--save-total-limit 3 \
--max_length 2000 \
--truncation_strategy delete

run sh: `/root/miniconda3/bin/python /root/miniconda3/lib/python3.12/site-packages/swift/cli/sft.py --model autodl-tmp/models/Qwen2.5-7B-Instruct --model_type qwen2_5 --dataset train_e.json --val_dataset valid_e.json --logging_steps 5 --save-steps 10 --learning_rate 3e-5 --output_dir output --lora_rank 8 --lora_alpha 16 --lora_dropout 0. --weight_decay 0.01 --num_train_epochs 5 --per_device_train_batch_size 2 --gradient_accumulation_steps 2 --warmup-ratio 0.01 --eval_steps 20 --evaluation-strategy steps --max-grad-norm 1.0 --lr-scheduler-type cosine --save-total-limit 3 --max_length 2000 --truncation_strategy delete`
[INFO:swift] Successfully registered `/root/miniconda3/lib/python3.12/site-packages/swift/llm/dataset/data/dataset_info.json`.
[INFO:swift] rank: -1, local_rank: -1, world_size: 1, local_world_size: 1
[INFO:swift] Loading the model using model_dir: autodl-tmp/models/Qwen2.5-7B-Instruct
[INFO:swift] Setting torch_dtype: torch.bfloat16
[INFO:swift] Because len(args.val_datas

## 用vllm推理部署

xxx 替换成训练后保存的 output/xxx/xxx

!!import !! 终端运行

In [ ]:
vllm serve models/Qwen2.5-7B-Instruct \
--enable-lora \
--lora-modules qwen2.5-lora-e=output/v1-20250405-134004/checkpoint-220 \
--max-model-len 2000 \
--dtype bfloat16 \
--device cuda

SyntaxError: invalid decimal literal (3121605136.py, line 1)

## 跑验证集数据


In [3]:
from openai import OpenAI

client = OpenAI(
    api_key="EMPTY",
    base_url="http://localhost:8000/v1",
)
client.models.list()

SyncPage[Model](data=[Model(id='autodl-tmp/models/Qwen2.5-7B-Instruct', created=1743833756, object='model', owned_by='vllm', root='autodl-tmp/models/Qwen2.5-7B-Instruct', parent=None, max_model_len=2000, permission=[{'id': 'modelperm-1b960b79f1e6498caf694f46c962919a', 'object': 'model_permission', 'created': 1743833756, 'allow_create_engine': False, 'allow_sampling': True, 'allow_logprobs': True, 'allow_search_indices': False, 'allow_view': True, 'allow_fine_tuning': False, 'organization': '*', 'group': None, 'is_blocking': False}]), Model(id='qwen2.5-lora-e', created=1743833756, object='model', owned_by='vllm', root='output/v1-20250405-134004/checkpoint-220', parent='autodl-tmp/models/Qwen2.5-7B-Instruct', max_model_len=None, permission=[{'id': 'modelperm-a876364ceb444ea59f3acd6272ed9c77', 'object': 'model_permission', 'created': 1743833756, 'allow_create_engine': False, 'allow_sampling': True, 'allow_logprobs': True, 'allow_search_indices': False, 'allow_view': True, 'allow_fine_tuni

In [4]:
import json

with open('results_e.json', 'r') as f:
    valid_data = json.load(f)
len(valid_data), valid_data[0]

(80,
 {'messages': [{'role': 'user',
    'content': '你是一个中文文本关系抽取领域的专家，请仔细分析以下文本，严格按以下要求执行：\n\n要求：\n一.识别所有的关系数据,有如下关系类型：游玩顺序,出行时间,停留时长,预算限制,人均预算,体验内容。\n二.关系对应的实体类型组合如下：\n    1. 体验内容：\n    - ob1 entity_label ∈ [目的地, 餐饮, 住宿]\n    - ob2 entity_label ∈ [产品]\n    2. 预算限制：\n   - ob1 entity_label ∈ [目的地, 餐饮, 住宿, 产品, 交通]\n   - ob2 entity_label ∈ [预算]\n    3. 人均预算：\n   - ob1 entity_label ∈ [预算]\n   - ob2 entity_label ∈ [人数]\n    4. 停留时长：\n   - ob1 entity_label ∈ [目的地, 餐饮, 住宿, 产品]\n   - ob2 entity_label ∈ [时长]\n    5. 出行时间：\n   - ob1 entity_label ∈ [出发地, 返程地, 目的地, 餐饮, 住宿, 产品]\n   - ob2 entity_label ∈ [时间]\n    6. 游玩顺序：\n   - ① ob1 entity_label ∈ [出发地]，ob2 实体类型 ∈ [目的地]\n   - ② ob1 entity_label ∈ [目的地]，ob2 实体类型 ∈ [目的地]\n   - ③ ob1 entity_label ∈ [目的地]，ob2 实体类型 ∈ [返程地]\n三.所有三元组必须遵循上述实体类型对\n四.所有实体必须来自输入的实体\n五.输出格式为JSON列表，每个元素包含三个字段："ob1", "rel", "ob2"\n\n输出示例：\n[\n  {"ob1": "x", "rel": "x", \'ob2\':\'x\'},\n  {"ob1": "x", "rel": "x", \'ob2\':\'x\'},\n]\n\n若无关系则输出:"找不到关系"\n\n请处理以下文本，已给出文本对应实体：\n\n文本:

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

prompt = """
你是一个中文文本关系抽取领域的专家，请仔细分析以下文本，严格按以下要求执行：

要求：
一.识别所有的关系数据,有如下关系类型：游玩顺序,出行时间,停留时长,预算限制,人均预算,体验内容。
二.关系对应的实体类型组合如下：
    1. 体验内容：
    - ob1 entity_label ∈ [目的地, 餐饮, 住宿]
    - ob2 entity_label ∈ [产品]
    2. 预算限制：
   - ob1 entity_label ∈ [目的地, 餐饮, 住宿, 产品, 交通]
   - ob2 entity_label ∈ [预算]
    3. 人均预算：
   - ob1 entity_label ∈ [预算]
   - ob2 entity_label ∈ [人数]
    4. 停留时长：
   - ob1 entity_label ∈ [目的地, 餐饮, 住宿, 产品]
   - ob2 entity_label ∈ [时长]
    5. 出行时间：
   - ob1 entity_label ∈ [出发地, 返程地, 目的地, 餐饮, 住宿, 产品]
   - ob2 entity_label ∈ [时间]
    6. 游玩顺序：
   - ① ob1 entity_label ∈ [出发地]，ob2 实体类型 ∈ [目的地]
   - ② ob1 entity_label ∈ [目的地]，ob2 实体类型 ∈ [目的地]
   - ③ ob1 entity_label ∈ [目的地]，ob2 实体类型 ∈ [返程地]
三.所有三元组必须遵循上述实体类型对
四.所有实体必须来自输入的实体
五.输出格式为JSON列表，每个元素包含三个字段："ob1", "rel", "ob2"

输出示例：
[
  {"ob1": "x", "rel": "x", 'ob2':'x'},
  {"ob1": "x", "rel": "x", 'ob2':'x'},
]

若无关系则输出:"找不到关系"

请处理以下文本，已给出文本对应实体：

{text}

""".strip()

def process(idx:int, msg: dict):
    api_key = "XXX"
    base_url = "http://localhost:8000/v1"
    # model = "models/Qwen2.5-7B-Instruct"
    model='qwen2.5-lora-e'
    # model = 'models/glm-4-9b-chat'
    # model = 'glm4-lora'
    # model = 'models/internlm3-8b-instruct'
    # model = 'internlm3-lora'
    temperature = 0.
    client = OpenAI(api_key=api_key, base_url=base_url)
    
    try:
        resp = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "user", "content": prompt.replace("{text}", msg["messages"][0]['content'])},
            ],
            temperature=temperature,
        )
        content = resp.choices[0].message.content
        msg[model] = content
    except:
        msg[model] = ''
    
    return idx, msg

max_workers = 60
start_time = time.time()
with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = [
        executor.submit(process, idx, msg) for idx, msg in enumerate(valid_data)
    ]

    num = 0
    print(f'>all {len(futures)} <<<<')
    for future in as_completed(futures):
        idx, msg = future.result()
        valid_data[idx] = msg

        if (num :=num + 1) and (num % max_workers==0 or num == len(futures)):
            print(f">> {time.strftime('%m-%d %H:%M:%S')} >> {num:<5} << {time.time()-start_time:.3f}s")

            with open('results.json', 'w', encoding='utf-8') as f:
                json.dump(valid_data, f, ensure_ascii=False, indent=2)
            start_time = time.time()

with open('results_e.json', 'w', encoding='utf-8') as f:
    json.dump(valid_data, f, ensure_ascii=False, indent=2)

>all 80 <<<<
>> 04-05 14:17:06 >> 60    << 26.604s
>> 04-05 14:17:11 >> 80    << 4.819s


In [11]:
valid_data[0].keys()

dict_keys(['messages', 'qwen2.5-lora-e', 'autodl-tmp/models/Qwen2.5-7B-Instruct'])

## 评估

In [12]:
import json

with open('results_e.json', 'r') as f:
    results = json.load(f)
results[0]

{'messages': [{'role': 'user',
   'content': '你是一个中文文本关系抽取领域的专家，请仔细分析以下文本，严格按以下要求执行：\n\n要求：\n一.识别所有的关系数据,有如下关系类型：游玩顺序,出行时间,停留时长,预算限制,人均预算,体验内容。\n二.关系对应的实体类型组合如下：\n    1. 体验内容：\n    - ob1 entity_label ∈ [目的地, 餐饮, 住宿]\n    - ob2 entity_label ∈ [产品]\n    2. 预算限制：\n   - ob1 entity_label ∈ [目的地, 餐饮, 住宿, 产品, 交通]\n   - ob2 entity_label ∈ [预算]\n    3. 人均预算：\n   - ob1 entity_label ∈ [预算]\n   - ob2 entity_label ∈ [人数]\n    4. 停留时长：\n   - ob1 entity_label ∈ [目的地, 餐饮, 住宿, 产品]\n   - ob2 entity_label ∈ [时长]\n    5. 出行时间：\n   - ob1 entity_label ∈ [出发地, 返程地, 目的地, 餐饮, 住宿, 产品]\n   - ob2 entity_label ∈ [时间]\n    6. 游玩顺序：\n   - ① ob1 entity_label ∈ [出发地]，ob2 实体类型 ∈ [目的地]\n   - ② ob1 entity_label ∈ [目的地]，ob2 实体类型 ∈ [目的地]\n   - ③ ob1 entity_label ∈ [目的地]，ob2 实体类型 ∈ [返程地]\n三.所有三元组必须遵循上述实体类型对\n四.所有实体必须来自输入的实体\n五.输出格式为JSON列表，每个元素包含三个字段："ob1", "rel", "ob2"\n\n输出示例：\n[\n  {"ob1": "x", "rel": "x", \'ob2\':\'x\'},\n  {"ob1": "x", "rel": "x", \'ob2\':\'x\'},\n]\n\n若无关系则输出:"找不到关系"\n\n请处理以下文本，已给出文本对应实体：\n\n文本:求推荐南京周边

- 计算acc pre f1

In [13]:
def calculate_metrics(preds:list[str], targets:list[str]):
    preds_set, targets_set = set(preds), set(targets)
    
    tp = len(preds_set & targets_set)  # True Positives
    fp = len(preds_set - targets_set)  # False Positives
    fn = len(targets_set - preds_set)  # False Negatives
    
    acc = tp / max(len(targets_set), len(preds_set)) if max(len(targets_set), len(preds_set)) > 0 else 0
    pre = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * pre * rec / (pre + rec) if (pre + rec) > 0 else 0
    
    return acc, pre,rec, f1


In [ ]:
models = [
    'models/Qwen2.5-7B-Instruct', 
    'qwen2.5-lora-e'
]
for model in models:
    scores = []
    enums = 0
    for d in results:
        t = d['messages'][1]['content']
        tt = [[i['ob1'], i['rel'],i['ob2']] for i in eval(t[7:-3])]
        targets = list(set(sum(tt, [])))

        try:
            jd = d[model].replace('null', '""').strip()
            if '```json' in jd:
                si = jd.index('```json')+7
            else:
                si = 0
            if '```' in jd[si:]:
                ei = jd.rindex('```')
            else:
                ei = len(jd)
            jd = jd[si:ei]
            
            tp = [[i['ob1'], i['rel'],i['ob2']] for i in eval(jd)]
            preds = list(set(sum(tp, [])))
            
            # 
            score = calculate_metrics(preds, targets)
            scores.append(score)
        except:
            enums +=1

    accs = [i[0] for i in scores]
    pres = [i[1] for i in scores]
    recs = [i[2] for i in scores]
    f1s  = [i[3] for i in scores]
    print('model:', model)
    print('enums:', enums)
    print('acc:', sum(accs)/len(accs))
    print('pre:', sum(pres)/len(pres))
    print('rec:', sum(recs)/len(recs))
    print('f1:', sum(f1s)/len(f1s))
    print('==')

model: autodl-tmp/models/Qwen2.5-7B-Instruct
enums: 5
acc: 0.6044064639064639
pre: 0.6055916490916491
rec: 0.890095238095238
f1: 0.697632258579627
==
model: qwen2.5-lora-e
enums: 1
acc: 0.8209495072683181
pre: 0.8930982519590115
rec: 0.8713111709211209
f1: 0.865133861517225
==


In [ ]:
#实验结果

In [ ]:
[{'instruction': '你是一个文本实体识别领域的专家，你需要从给定的句子中提取目的地;出发地;返程地;餐饮;住宿;产品;交通;预算;时长;时间;人群;强度;人气;人数;天气.以json格式输出,如{"entity_text":"北京","entity_label":"目的地"}注意:1.输出的每一行都必须是正确的json字符串.2.找不到任何实体时,输出"没有找到任何实体".',
  'input': '文本:五一，我要去泉州旅游，想要感受一下宋元时期中国的世界海洋商贸盛况，计划去3天2晚，住宿预算大概是300~400元每晚，希望住在市中心，能帮我规划一下路线吗?',
  'output': '{"entity_text":"泉州","entity_label":"目的地"}{"entity_text":"市中心","entity_label":"住宿"}{"entity_text":"300~400元","entity_label":"预算"}{"entity_text":"五一","entity_label":"时间"}{"entity_text":"3天2晚","entity_label":"时长"}{"entity_text":"宋元时期中国的世界海洋商贸盛况","entity_label":"产品"}'}]

In [55]:
from openai import OpenAI

client = OpenAI(
    api_key="EMPTY",
    base_url="http://localhost:8000/v1",
)
client.models.list()

SyncPage[Model](data=[Model(id='autodl-tmp/models/Qwen2.5-7B-Instruct', created=1743377326, object='model', owned_by='vllm', root='autodl-tmp/models/Qwen2.5-7B-Instruct', parent=None, max_model_len=2000, permission=[{'id': 'modelperm-b6c8b5aae64c45d98c61e0292080ce91', 'object': 'model_permission', 'created': 1743377326, 'allow_create_engine': False, 'allow_sampling': True, 'allow_logprobs': True, 'allow_search_indices': False, 'allow_view': True, 'allow_fine_tuning': False, 'organization': '*', 'group': None, 'is_blocking': False}]), Model(id='qwen2.5-lora-e', created=1743377326, object='model', owned_by='vllm', root='output/v0-20250330-034516/checkpoint-158', parent='autodl-tmp/models/Qwen2.5-7B-Instruct', max_model_len=None, permission=[{'id': 'modelperm-439e45d238ed4ec98ec40c05a9fd287a', 'object': 'model_permission', 'created': 1743377326, 'allow_create_engine': False, 'allow_sampling': True, 'allow_logprobs': True, 'allow_search_indices': False, 'allow_view': True, 'allow_fine_tuni

In [56]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

prompt = """
你是一个中文文本关系抽取领域的专家，请仔细分析以下文本，严格按以下要求执行：

要求：
一.识别所有的关系数据,有如下关系类型：游玩顺序,出行时间,停留时长,预算限制,人均预算,体验内容。
二.关系对应的实体类型组合如下：
    1. 体验内容：
    - ob1 entity_label ∈ [目的地, 餐饮, 住宿]
    - ob2 entity_label ∈ [产品]
    2. 预算限制：
   - ob1 entity_label ∈ [目的地, 餐饮, 住宿, 产品, 交通]
   - ob2 entity_label ∈ [预算]
    3. 人均预算：
   - ob1 entity_label ∈ [预算]
   - ob2 entity_label ∈ [人数]
    4. 停留时长：
   - ob1 entity_label ∈ [目的地, 餐饮, 住宿, 产品]
   - ob2 entity_label ∈ [时长]
    5. 出行时间：
   - ob1 entity_label ∈ [出发地, 返程地, 目的地, 餐饮, 住宿, 产品]
   - ob2 entity_label ∈ [时间]
    6. 游玩顺序：
   - ① ob1 entity_label ∈ [出发地]，ob2 实体类型 ∈ [目的地]
   - ② ob1 entity_label ∈ [目的地]，ob2 实体类型 ∈ [目的地]
   - ③ ob1 entity_label ∈ [目的地]，ob2 实体类型 ∈ [返程地]
三.所有三元组必须遵循上述实体类型对
四.所有实体必须来自输入的实体
五.输出格式为JSON列表，每个元素包含三个字段："ob1", "rel", "ob2"

输出示例：
[
  {"ob1": "x", "rel": "x", 'ob2':'x'},
  {"ob1": "x", "rel": "x", 'ob2':'x'},
]

若无关系则输出:"找不到关系"

请处理以下文本，已给出文本对应实体：

{text}

""".strip()

def process(idx, msg):
    # 假设 API 密钥和基本 URL
    api_key = "XXX"
    base_url = "http://localhost:8000/v1"
    model = 'qwen2.5-lora-e'
    temperature = 0.
    
    try:
        formatted_prompt = prompt.replace("{text}", msg)
        resp = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": formatted_prompt}],
            temperature=temperature,
        )
        content = resp.choices[0].message.content
        return idx, content
    except Exception as e:
        print(f"Error processing index {idx}: {e}")
        return idx, f"Error: {str(e)}"

# 示例输入数据
valid_data = [{
    "messages": [{
        "content": "文本:五一，我要去泉州旅游，想要感受一下宋元时期中国的世界海洋商贸盛况，计划去3天2晚，住宿预算大概是300~400元每晚，希望住在市中心，能帮我规划一下路线吗?\n实体:{\"entity_text\":\"泉州\",\"entity_label\":\"目的地\"}{\"entity_text\":\"市中心\",\"entity_label\":\"住宿\"}{\"entity_text\":\"300~400元\",\"entity_label\":\"预算\"}{\"entity_text\":\"五一\",\"entity_label\":\"时间\"}{\"entity_text\":\"3天2晚\",\"entity_label\":\"时长\"}{\"entity_text\":\"宋元时期中国的世界海洋商贸盛况\",\"entity_label\":\"产品\"}"
    }]
}]

max_workers = 1  # 根据实际需要设置线程数
with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = [
        executor.submit(process, idx, msg["messages"][0]['content']) for idx, msg in enumerate(valid_data)
    ]

    results = {}
    for future in as_completed(futures):
        idx, result = future.result()
        results[idx] = result

# 输出结果
print(results)

# 将结果写入文件
with open('try.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

{0: '```json\n[\n  {\n    "ob1": "泉州",\n    "rel": "停留时长",\n    "ob2": "3天2晚"\n  },\n  {\n    "ob1": "泉州",\n    "rel": "预算限制",\n    "ob2": "300~400元"\n  }\n]\n```'}


In [57]:
import json

with open('try.json', 'r') as f:
    results = json.load(f)
results[0]

KeyError: 0

In [ ]:
[{"ob1": "泉州","rel": "停留时长","ob2": "3天2晚"},{"ob1": "市中心","rel": "预算限制","ob2": "300~400元"}]